In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [2]:
df=pd.read_csv('loan_data.csv')

In [3]:
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [4]:
x = df.drop(columns='loan_status')
y = df.loan_status

In [5]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=42, train_size=0.8)

In [6]:
# num_col = x.select_dtypes(include = 'number').columns
obj_col = x.select_dtypes(include = 'object').columns

In [7]:
x[obj_col].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [8]:
x['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [9]:
order = ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [10]:
preprocessing  = ColumnTransformer(
    transformers=[
        ('onehot_encodedr', OneHotEncoder(handle_unknown='ignore'), obj_col.drop('person_education')),
        ('orndinal_encoder', OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1), ['person_education'])

    ], remainder= 'passthrough'
)

main_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

grid_search_cv = GridSearchCV(
    estimator = main_pipeline,
    param_grid={
        'model__criterion' : ['gini', 'entropy'],           # if we are using any algorithm then there is no need to use that model name but if we are using pipeline then we have to use model name along with double underscore
        'model__max_depth' : [None, 5, 20, 50, 100],        # e.g., model__parameter
        'model__min_samples_split' : [2, 5, 7, 10],
        'model__min_samples_leaf' : [1, 3, 5, 7, 10],
        'model__splitter' : ['best', 'random']
    }, verbose = 10, n_jobs = -1
)

grid_search_cv.fit(xtrain, ytrain)

Fitting 5 folds for each of 400 candidates, totalling 2000 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,10
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot_encodedr', ...), ('orndinal_encoder', ...)]"


In [11]:
grid_search_cv.best_estimator_

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot_encodedr', ...), ('orndinal_encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 20,
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [13]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.43770719, 0.20653162, 0.39206185, 0.16297479, 0.38290796,
        0.17159753, 0.38937311, 0.17455502, 0.36986427, 0.16326575,
        0.35496202, 0.15793848, 0.35963144, 0.16487384, 0.34770403,
        0.16133766, 0.34784365, 0.16077957, 0.35024128, 0.15286226,
        0.36815124, 0.21557393, 0.40298238, 0.17274637, 0.3545805 ,
        0.15996051, 0.41652722, 0.18996468, 0.33331871, 0.15188279,
        0.33014622, 0.16629949, 0.33700228, 0.14668021, 0.33834205,
        0.15195527, 0.3341094 , 0.15291691, 0.32222118, 0.15697594,
        0.1865715 , 0.11539116, 0.17422299, 0.11863413, 0.18312373,
        0.12724128, 0.18541799, 0.11434426, 0.2011878 , 0.13166237,
        0.18571849, 0.12612505, 0.18880916, 0.11562076, 0.18154216,
        0.12837396, 0.19479847, 0.12173333, 0.19730558, 0.12659664,
        0.20076485, 0.11280661, 0.18185468, 0.11754913, 0.20307703,
        0.1210948 , 0.18432913, 0.11736798, 0.18327589, 0.12708907,
        0.18292341, 0.1233798 ,

In [14]:
res = pd.DataFrame(grid_search_cv.cv_results_)

In [15]:
res.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
316,0.453242,0.057475,0.032881,0.005677,entropy,20,10,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
318,0.433866,0.014639,0.032086,0.001250,entropy,20,10,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
312,0.547783,0.049314,0.032790,0.003970,entropy,20,10,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
314,0.435626,0.015440,0.033343,0.003248,entropy,20,10,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
78,0.183746,0.023559,0.026900,0.004686,gini,5,10,10,best,"{'model__criterion': 'gini', 'model__max_depth...",0.909861,0.916667,0.908889,0.915556,0.915417,0.913278,0.003231,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0.123380,0.014016,0.025667,0.005946,gini,5,7,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872639,0.872417,0.000461,393
79,0.110880,0.014835,0.027622,0.003297,gini,5,10,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
77,0.117713,0.005691,0.027186,0.003580,gini,5,10,7,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
75,0.123999,0.009748,0.023044,0.007163,gini,5,10,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
